# Quantum Phase Estimation
Given a unitary $U$ and one of its eigenstates $|\psi\rangle$ with $U|\psi\rangle=e^{2\pi i\varphi}|\psi\rangle$, QPE reads off the phase $\varphi$.
A counting register picks up the phase via controlled-$U^{2^j}$ (phase kickback), then an **inverse QFT** turns those phases into a plain binary number.
Here $U=P(2\pi\varphi)$ with eigenstate $|1\rangle$, so the measured integer divided by $2^n$ estimates $\varphi$.
Run twice — an ideal simulator and a noisy one carrying a real IBM device's noise model (offline).

In [ ]:
# QFT builder (we need its inverse for QPE)
from math import pi
from qiskit import QuantumCircuit

def qft(n: int) -> QuantumCircuit:
    qc = QuantumCircuit(n)
    def rotations(k):
        if k == 0:
            return
        k -= 1
        qc.h(k)
        for q in range(k):
            qc.cp(pi / 2 ** (k - q), q, k)
        rotations(k)
    rotations(n)
    for i in range(n // 2):
        qc.swap(i, n - 1 - i)
    return qc

In [ ]:
# Assemble QPE: n counting qubits + 1 eigenstate qubit, then draw it
phi = 5 / 16                                 # true phase to estimate (exact in 4 bits)
n = 4                                        # counting precision (bits)

qc = QuantumCircuit(n + 1, n)
qc.x(n)                                      # eigenstate |1> of P(theta)
qc.h(range(n))                               # counting register into superposition
for j in range(n):
    qc.cp(2 * pi * phi * (2 ** j), j, n)     # controlled-U^{2^j}: kick phase into counting bit j
qc.barrier()
qc.compose(qft(n).inverse(), qubits=range(n), inplace=True)   # inverse QFT: phases -> binary
qc.measure(range(n), range(n))

qc.draw("mpl")

In [ ]:
# Run on an ideal simulator and a noisy one (FakeSherbrooke = real IBM device noise model, offline)
from qiskit import transpile
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

ideal = AerSimulator()
noisy = AerSimulator.from_backend(FakeSherbrooke())

counts_ideal = ideal.run(transpile(qc, ideal), shots=1024).result().get_counts()
counts_noisy = noisy.run(transpile(qc, noisy, optimization_level=3), shots=1024).result().get_counts()

In [ ]:
# Read the peak as integer / 2^n to recover the phase
from qiskit.visualization import plot_histogram

peak_ideal = max(counts_ideal, key=counts_ideal.get)
peak_noisy = max(counts_noisy, key=counts_noisy.get)
print("true phi     :", phi)
print("ideal est    :", int(peak_ideal, 2), "/", 2 ** n, "=", int(peak_ideal, 2) / 2 ** n)
print("noisy est    :", int(peak_noisy, 2), "/", 2 ** n, "=", int(peak_noisy, 2) / 2 ** n)
plot_histogram([counts_noisy, counts_ideal], legend=["noisy", "ideal"])

## What the results mean

**Peak at `0101` = 5, phase = 5/16.** The eigenstate never changes — $P(\theta)|1\rangle=e^{i\theta}|1\rangle$ leaves it as $|1\rangle$. The phase $e^{2\pi i\varphi}$ instead **kicks back** onto each counting qubit, and doubling the rotation on qubit $j$ (the $U^{2^j}$) writes successive binary digits of $\varphi$ into the register's phases. The inverse QFT is the reverse of the QFT notebook: it collapses that phase pattern back into a single basis state whose bits spell the number. Divide by $2^n$ and you get $\varphi$.

**Exact vs. approximate.** $5/16 = 0.0101_2$ fits in 4 bits, so the peak is sharp and the estimate is exact. Pick a $\varphi$ that doesn't fit (e.g. `0.3`) and the peak lands on the nearest 4-bit value with a spread around it — more counting qubits $n$ buy more precision. That is the whole knob: $n$ = bits of $\varphi$.

**Why this is the keystone.** QPE = the eigenvalue-reading machine. Shor's factoring is QPE on the modular-multiplication unitary; HHL and quantum chemistry lean on it too. The pattern is always the same three moves you see here: superpose the counting register, kick the phase in with controlled powers of $U$, inverse-QFT to read it.

**Ideal vs. noisy.** This circuit is deeper than anything in Phase 1 — controlled-phase powers plus a full inverse QFT means many two-qubit gates after transpilation. The ideal run nails `0101` exactly; the noisy run (a real IBM device's error model, applied offline) keeps a peak at `0101` but with a wider skirt, and the peak can slip to `0100` or `0110`, costing a bit of precision. That fragility is exactly why near-term QPE stays shallow and Shor needs error correction.